In [9]:
import getpass
import json
from typing import Annotated, Sequence, TypedDict, Literal
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# 1. Securely ingest authorization strings directly into volatile memory
anthropic_api_key = getpass.getpass("Enter your Anthropic API Key: ")
openai_api_key = getpass.getpass("Enter your OpenAI API Key: ")
serper_api_key = getpass.getpass("Enter your Serper API Key string: ")

print("\n✅ Keys captured! Operating cleanly within isolation environment parameters.")


✅ Keys captured! Operating cleanly within isolation environment parameters.


In [10]:
@tool
def google_search(query: str) -> str:
    """Queries the live internet to find matching webpage links and snippets for a given search query."""
    print(f"\n[TOOL CALLED -> GOOGLE SEARCH]: Running query for '{query}'")
    if "langgraph memory" in query.lower():
        return json.dumps({
            "results": [
                {"title": "LangGraph Memory Management", "url": "https://langchain.com/graph-memory", "snippet": "LangGraph uses checkpointers like MemorySaver to persist state across threads natively."},
                {"title": "State Persistence Tutorial", "url": "https://docs.langchain.com/time-travel", "snippet": "Time travel features allow developers to rewind state history to any historical checkpoint."}
            ]
        })
    return "Search yielded no critical anomalies. Target domain appears stable."

@tool
def fetch_webpage_content(url: str) -> str:
    """Downloads and extracts the raw textual body content of a specific webpage URL."""
    print(f"\n[TOOL CALLED -> FETCH WEBPAGE]: Navigating to raw resource path: {url}")
    if "graph-memory" in url:
        return "Deep Dive Content: LangGraph's MemorySaver acts as an in-memory transactional save-game engine. It serializes the State dictionary after every node transition using specific thread configurations."
    return "Standard webpage landing interface. Content unverified."

@tool
def query_vector_database(semantic_terms: str) -> str:
    """Searches our private internal vector database using semantic terms to find historical internal enterprise documentation."""
    print(f"\n[TOOL CALLED -> VECTOR DB]: Searching internal vector index for semantic concepts: '{semantic_terms}'")
    if "security" in semantic_terms.lower() or "policy" in semantic_terms.lower():
        return "Internal Document Ref #402: Enterprise compliance rules dictate that all automated infrastructure updates require a human checkpoint confirmation before hitting production nodes."
    return "No matching internal vector segments found for the query vector slice."

# Map identifiers explicitly
tool_registry = {
    "google_search": google_search,
    "fetch_webpage_content": fetch_webpage_content,
    "query_vector_database": query_vector_database
}
tools_list = [google_search, fetch_webpage_content, query_vector_database]

In [11]:
# Initialize LLM model using the variable string key directly
# Replace Cell 3 model initialization with this if you want to use Claude 3.5 Sonnet:
llm_model = ChatAnthropic(
    model="claude-sonnet-4-6", 
    temperature=0, 
    anthropic_api_key=anthropic_api_key
).bind_tools(tools_list)

# Define Graph State
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# Construct Nodes
def researcher_brain_node(state: AgentState) -> dict:
    print("\n🧠 [NODE -> AGENT BRAIN]: Planning next step based on history...")
    response = llm_model.invoke(state["messages"])
    return {"messages": [response]}

def tool_execution_node(state: AgentState) -> dict:
    print("\n🛠️ [NODE -> ACTION TOOLS]: Running requested tool tools...")
    last_message = state["messages"][-1]
    tool_outputs = []
    
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        target_tool = tool_registry[tool_name]
        
        observation = target_tool.invoke(tool_args)
        
        tool_message = ToolMessage(
            content=str(observation),
            tool_call_id=tool_call["id"],
            name=tool_name
        )
        tool_outputs.append(tool_message)
        
    return {"messages": tool_outputs}

# Router Logic
def should_continue_router(state: AgentState) -> Literal["call_tools", "exit_pipeline"]:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        print(f"🔀 [ROUTER]: Tool calls requested -> {[tc['name'] for tc in last_message.tool_calls]}")
        return "call_tools"
    print("🏁 [ROUTER]: Mission target achieved. Exiting graph pipeline loops.")
    return "exit_pipeline"

In [14]:
from langgraph.checkpoint.memory import MemorySaver

workflow_builder = StateGraph(AgentState)
workflow_builder.add_node("agent_brain", researcher_brain_node)
workflow_builder.add_node("action_tools", tool_execution_node)

workflow_builder.add_edge(START, "agent_brain")
workflow_builder.add_conditional_edges(
    "agent_brain",
    should_continue_router,
    {
        "call_tools": "action_tools",
        "exit_pipeline": END
    }
)
workflow_builder.add_edge("action_tools", "agent_brain")

# FIX: Add an in-memory checkpointer during compilation
agent_memory = MemorySaver()
research_agent_app = workflow_builder.compile(checkpointer=agent_memory)

print("🎉 StateGraph compiled completely with memory checkpointing enabled.")

🎉 StateGraph compiled completely with memory checkpointing enabled.


In [15]:
user_objective = (
    "Find out how LangGraph handles memory by searching the internet. "
    "Then, pull details from the specific URL you find. Finally, check our internal "
    "vector database compliance policies regarding manual checkpoints."
)

initial_state_input = {"messages": [HumanMessage(content=user_objective)]}

# FIX: Define a thread configuration dictionary
research_config = {"configurable": {"thread_id": "research_session_01"}}

print("🚀 Launching Autonomous Re-Act Mission...")

# FIX: Pass the config dictionary to the stream function
events = research_agent_app.stream(initial_state_input, config=research_config, stream_mode="updates")

for event in events:
    for node_name, state_update in event.items():
        print(f"\n🛑 [PAUSED] Node '{node_name}' finished execution step.")
        
        action = input("\nPress Enter to ALLOW the agent to evaluate the observation and plan next steps (or type 'quit'): ")
        if action.strip().lower() == 'quit':
            print("❌ Workflow halted by manual request.")
            break

# FIX: Pass the same thread config to get_state
final_state = research_agent_app.get_state(config=research_config)
print("\n" + "=" * 70)
print("🏆 FINAL CONSOLIDATED RESEARCH BRIEFING REPORT")
print("=" * 70)
print(final_state.values["messages"][-1].content)

🚀 Launching Autonomous Re-Act Mission...

🧠 [NODE -> AGENT BRAIN]: Planning next step based on history...
🔀 [ROUTER]: Tool calls requested -> ['google_search', 'query_vector_database']

🛑 [PAUSED] Node 'agent_brain' finished execution step.

🛠️ [NODE -> ACTION TOOLS]: Running requested tool tools...

[TOOL CALLED -> GOOGLE SEARCH]: Running query for 'LangGraph memory handling'

[TOOL CALLED -> VECTOR DB]: Searching internal vector index for semantic concepts: 'compliance policies manual checkpoints'

🛑 [PAUSED] Node 'action_tools' finished execution step.

🧠 [NODE -> AGENT BRAIN]: Planning next step based on history...
🔀 [ROUTER]: Tool calls requested -> ['fetch_webpage_content']

🛑 [PAUSED] Node 'agent_brain' finished execution step.

🛠️ [NODE -> ACTION TOOLS]: Running requested tool tools...

[TOOL CALLED -> FETCH WEBPAGE]: Navigating to raw resource path: https://langchain.com/graph-memory

🛑 [PAUSED] Node 'action_tools' finished execution step.

🧠 [NODE -> AGENT BRAIN]: Planning ne